# Prédiction des charges d’assurance – Analyse exploratoire (EDA)

## Contexte
Un assureur souhaite anticiper les charges médicales à partir de variables
démographiques et de santé afin d’améliorer la tarification et la gestion du risque.

## Objectifs de ce notebook
- Comprendre la structure et la qualité des données
- Identifier les variables influençant les charges
- Mettre en évidence des insights métier
- Préparer les choix de modélisation pour la semaine 2


### Dictionnaire des Variables

| Variable | Type | Rôle | Description |
| :--- | :--- | :--- | :--- |
| **age** | Numérique | Feature | Âge de l'assuré (18-64 ans) |
| **sex** | Catégorielle | Feature | Sexe (male/female) |
| **bmi** | Numérique | Feature | Indice de Masse Corporelle (IMC) |
| **children** | Numérique | Feature | Nombre d'enfants à charge |
| **smoker** | Catégorielle | Feature | Fumeur (yes/no) |
| **region** | Catégorielle | Feature | Région aux USA (4 zones) |
| **charges** | Numérique | **Cible** | Frais médicaux annuels |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import plotly.express as px
from scipy.stats import chi2_contingency
import scipy.stats as stats

# 1. Chargement des données (Chemin pour dossier /notebooks)
path = '../data/insurance.csv'

if os.path.exists(path):
    # Création d'un dictionnaire pour forcer le typage (Par défaut Pandas attribue toujours le type de données à la mémoire la plus élevée)
    df_without_optimized = pd.read_csv(path, low_memory=True)
    dtype_dict = {'age': 'int8', 'sex': 'category', 'bmi': 'float16', 'children': 'int8', 'smoker': 'category', 'region': 'category', 'charges': 'float32'}

    df = pd.read_csv(path, low_memory=True, dtype=dtype_dict)
    df["smoker"] = df["smoker"].map({"yes": 1, "no": 0}).astype("bool")

    print("Avant optimisation :")
    print("Utilisation mémoire par colonne :")
    print(f"  Niveau : {df_without_optimized.memory_usage(deep=True)} bytes")

    print(f"\nTotal : {df_without_optimized.memory_usage(deep=True).sum() / 1024:.2f} KB")
    print("\nAprès optimisation :")
    print("Utilisation mémoire par colonne :")
    print(f"  Niveau : {df.memory_usage(deep=True)} bytes")

    print("Utilisation mémoire par colonne :")
    
    print(f"\nTotal : {df.memory_usage(deep=True).sum() / 1024:.2f} KB")
    print(" Les données sont prêtes pour l'analyse de la Semaine 1.")
    lines, cols = df.shape
    print(f"Dataset : {lines} lignes et {cols} colonnes.")
else:
    print(f" Erreur : Je ne trouve pas le fichier au chemin : {path}")
    print("Vérifie bien que ton notebook est dans  le dossier 'notebooks'.")


df.isna().sum()

In [ ]:
# Fixation de la graine aléatoire pour la reproductibilité
import numpy as np
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Graine aléatoire fixée à : {RANDOM_SEED}")

In [ ]:
# 3 Informations sur le DataFrame
# contrôle des doublons
number_duplicated = df.duplicated().sum()
print(f"Il y a {number_duplicated} doublon")
# Visualiser le doublon
print(df[df.duplicated()])

# Voir toutes les lignes concernées
print(df[df.duplicated(keep=False)])
# Vérifier doublons sur les colonnes clés
df.duplicated(subset=["age","sex","bmi","children","smoker","region"]).sum()

df[df.duplicated(keep=False)]

# les lignes concerné sont strictement identique, nous avons donc choisi de le supprimer
# Suppression des doublons
df = df.drop_duplicates()

# Vérification après suppression
print(df.duplicated().sum())


# Vérification de valeur null
print(df.isnull().sum())

print("Aucune valeur null")



In [ ]:
# Statistiques de base : mean, median, std, min, max, quartiles
df.describe()

In [ ]:
# Nombre de valeurs uniques par colonne
df.nunique()



# 2. Analyse Univariée (Distributions)

Analyse de chaque variable indépendamment pour comprendre leur répartition et détecter d'éventuels problèmes (skewness, outliers).

## 2.1 Variables numériques

### Les charges

In [ ]:
print(df["charges"].describe())

In [ ]:
df["charges"].skew()

In [ ]:

fig = px.histogram(
    df,
    x="charges",
    nbins=50,
    opacity=0.75,
    title="Distribution des charges d'assurance"
)

fig.update_layout(
    xaxis_title="Charges",
    yaxis_title="Fréquence",
    bargap=0.1
)

fig.show()

In [ ]:
plt.figure(figsize=(8,3))
sns.boxplot(x=df["charges"])
plt.title("Boxplot des charges d'assurance")
plt.show()


In [ ]:
q1 = df["charges"].quantile(0.25)
q3 = df["charges"].quantile(0.75)
iqr = q3 - q1

q1, q3, iqr


La distribution des charges est fortement asymétrique à droite.
La majorité des observations se concentre sur des montants relativement faibles,
tandis qu’un nombre limité d’assurés présente des charges très élevées.

La moyenne est nettement supérieure à la médiane, ce qui confirme
l’influence importante des valeurs extrêmes sur la distribution.

Le boxplot met en évidence de nombreux outliers supérieurs,
et l’écart interquartile élevé traduit une forte dispersion des coûts.


D’un point de vue métier, cette distribution reflète un phénomène classique
en assurance santé : une minorité d’assurés concentre une part significative
des dépenses médicales.

Ces profils à coûts élevés représentent un risque financier majeur
pour l’assureur et justifient une attention particulière dans
les modèles de tarification et de gestion du risque.


In [ ]:
print(df["charges"].mean().round(2))

In [ ]:
df["charges"].skew()

Le coefficient d’asymétrie est élevé, confirmant une distribution
fortement déséquilibrée vers les charges élevées.


### Les Ages

In [ ]:
df["age"].describe()

In [ ]:
df["age"].skew()

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df["age"], kde=True)
plt.title("Distribution de l'âge des assurés")
plt.xlabel("Âge")
plt.ylabel("Fréquence")
plt.show()


In [ ]:
plt.figure(figsize=(8,3))
sns.boxplot(x=df["age"])
plt.title("Boxplot de l'âge")
plt.show()


La distribution de l’âge est relativement uniforme sur l’intervalle observé,
avec une légère concentration autour des âges moyens.
La moyenne et la médiane sont proches, indiquant une distribution
globalement symétrique, sans asymétrie marquée.

Le boxplot ne met pas en évidence de valeurs extrêmes significatives.


D’un point de vue métier, la population assurée couvre un large éventail d’âges,
ce qui permet au modèle d’apprendre des profils de risque variés.
L’âge est un facteur naturellement lié à la consommation de soins
et constitue une variable explicative pertinente pour la prédiction des charges.


La variable `age` ne nécessite pas de traitement particulier concernant
les valeurs extrêmes. Une standardisation pourra néanmoins être appliquée
afin d’assurer une mise à l’échelle cohérente avec les autres variables numériques.


### Le BMI

In [ ]:
df["bmi"].describe()

In [ ]:
df["bmi"].skew()

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df["bmi"], kde=True)
plt.title("Distribution du BMI")
plt.xlabel("BMI")
plt.ylabel("Fréquence")
plt.show()

In [ ]:
plt.figure(figsize=(8,3))
sns.boxplot(x=df["bmi"])
plt.title("Boxplot du BMI")
plt.show()

La distribution du BMI est légèrement asymétrique à droite,
avec une concentration autour des valeurs correspondant
au surpoids et à l’obésité légère.

Quelques valeurs extrêmes sont observées, mais elles restent
cohérentes d’un point de vue médical.


D’un point de vue métier, un BMI élevé est généralement associé
à un risque accru de pathologies chroniques, entraînant
des coûts médicaux plus importants.

La présence de BMI élevés dans la population étudiée
suggère un levier de segmentation pertinent pour l’assureur.


La variable `bmi` pourra être conservée telle quelle,
avec une standardisation. Une interaction avec d’autres variables
(comme le tabagisme ou l’âge) pourra être explorée ultérieurement.

### les Enfants

In [ ]:
df["children"].describe()

In [ ]:
print(df["children"].skew())

In [ ]:
children_counts = (
    df.groupby("children", observed=True)
      .size()
      .reset_index(name="count")
)

px.bar(
    children_counts,
    x="children",
    y="count",
    title="Répartition des assurés par sexe"
).show()

In [ ]:
plt.figure(figsize=(8,3))
sns.boxplot(x=df["children"])
plt.title("Boxplot du nombre d'enfants")
plt.show()

La distribution du nombre d’enfants est discrète et fortement concentrée
sur les valeurs faibles (0, 1 ou 2 enfants).
La majorité des assurés n’a pas ou peu d’enfants à charge.

La variable présente une asymétrie à droite,
avec très peu d’observations pour les valeurs élevées.


D’un point de vue métier, le nombre d’enfants à charge peut influencer
les charges globales d’un foyer, mais son impact individuel
sur les coûts médicaux reste potentiellement limité.

Cette variable pourrait néanmoins jouer un rôle secondaire
dans la prédiction des charges.


La variable `children`, bien que numérique, est discrète.
Elle pourra être utilisée telle quelle dans un modèle linéaire,
sans transformation particulière.


## 2.2 Variables catégorielles

### le Sex

In [ ]:
df["sex"].value_counts()

In [ ]:
df["sex"].value_counts(normalize=True) * 100

In [ ]:
sex_counts = (
    df.groupby("sex", observed=True)
      .size()
      .reset_index(name="count")
)


px.bar(
    sex_counts,
    x="sex",
    y="count",
    title="Répartition des assurés par sexe"
).show()



La population assurée est relativement équilibrée entre hommes et femmes,
avec une légère majorité d’hommes.
Aucun déséquilibre majeur n’est observé entre les deux modalités.


D’un point de vue métier, cette répartition équilibrée permet
d’évaluer l’impact du sexe sur les charges sans biais lié
à une surreprésentation d’une catégorie.


La variable `sex` pourra être intégrée au modèle via un encodage
binaire (One-Hot Encoding).


### Fumeurs ou non

In [ ]:
df["smoker"].value_counts()

In [ ]:
df["smoker"].value_counts(normalize=True) * 100

In [ ]:
smoker_counts = (
    df.groupby("smoker_label", observed=True)
      .size()
      .reset_index(name="count")
)

px.bar(
    smoker_counts,
    x="smoker_label",
    y="count",
    title="Répartition des assurés selon le statut fumeur"
).show()


La majorité des assurés sont non-fumeurs, tandis que les fumeurs
représentent une minorité de la population étudiée.


D’un point de vue métier, bien que les fumeurs soient minoritaires,
ils sont généralement associés à un risque médical plus élevé,
ce qui laisse présager un impact significatif sur les charges d’assurance.


La variable `smoker` est susceptible d’être un facteur explicatif majeur.
Elle sera encodée et analysée avec attention lors de la modélisation.


### Les Regions

In [ ]:
df["region"].value_counts()

In [ ]:
df["region"].value_counts(normalize=True) * 100

In [ ]:
region_counts = (
    df.groupby("region", observed=True)
      .size()
      .reset_index(name="count")
)

px.bar(
    region_counts,
    x="region",
    y="count",
    title="Répartition des assurés par région"
).show()


La population assurée est relativement bien répartie entre les différentes régions,
sans déséquilibre majeur entre les modalités.


D’un point de vue métier, l’absence de déséquilibre régional suggère
que la variable `region` ne reflète pas une segmentation forte
des profils de risque à ce stade de l’analyse.


La variable `region` pourra être intégrée via un encodage One-Hot.
Son impact réel sera évalué lors de la phase de modélisation.


# 3 Analyse bivariée

## 3.1 Variables catégorielles

### Smoker vs charges

Objectif :
Comparer les charges d’assurance entre fumeurs et non-fumeurs
afin d’évaluer l’impact du tabagisme sur les coûts médicaux.


In [ ]:
df.groupby("smoker")["charges"].describe()

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x="smoker_label", y="charges", data=df)
plt.title("Charges d'assurance selon le statut fumeur")
plt.xlabel("Statut fumeur")
plt.ylabel("Charges")
plt.show()


Les fumeurs présentent des charges nettement plus élevées que les non-fumeurs,
avec une médiane et une moyenne largement supérieures.

La dispersion des charges est également beaucoup plus importante
chez les fumeurs, indiquant une variabilité accrue des coûts.


D’un point de vue métier, le tabagisme apparaît comme le principal
facteur de risque identifié dans le dataset.

Bien que les fumeurs soient minoritaires, ils génèrent des coûts
médicaux significativement plus élevés, ce qui justifie une
segmentation tarifaire spécifique.


La variable `smoker` sera un facteur explicatif clé du modèle.
Des interactions avec l’âge ou le BMI pourront être explorées
lors des phases ultérieures.


### Sex vs charges

Objectif :
Comparer les charges d’assurance entre hommes et femmes
afin d’identifier d’éventuelles différences de coûts.


In [ ]:
df.groupby("sex")["charges"].describe()


In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x="sex", y="charges", data=df)
plt.title("Charges d'assurance selon le sexe")
plt.xlabel("Sexe")
plt.ylabel("Charges")
plt.show()


Les distributions des charges entre hommes et femmes sont proches.
Les médianes sont similaires et les écarts observés restent modérés
comparativement à d’autres variables comme le tabagisme.

D’un point de vue métier, le sexe ne semble pas être un facteur
discriminant majeur des charges dans ce dataset.

Son impact potentiel devra néanmoins être évalué dans le modèle final.


La variable `sex` pourra être intégrée au modèle,
mais son poids explicatif est attendu comme secondaire.


### region vs charges

Objectif :
Comparer les charges d’assurance entre les différentes régions
afin d’identifier d’éventuelles disparités géographiques.


In [ ]:
df.groupby("region")["charges"].describe()


In [ ]:
plt.figure(figsize=(8,4))
sns.boxplot(x="region", y="charges", data=df)
plt.title("Charges d'assurance selon la région")
plt.xlabel("Région")
plt.ylabel("Charges")
plt.show()


In [ ]:
pd.crosstab(df["region"], df["smoker_label"], normalize="index") * 100

In [ ]:
ct = pd.crosstab(df["region"], df["smoker_label"], normalize="index")

ct.plot(kind="bar", stacked=True, figsize=(8,5))
plt.title("Répartition des fumeurs par région")
plt.xlabel("Région")
plt.ylabel("Proportion")
plt.legend(title="Statut fumeur")
plt.show()


Les distributions des charges sont relativement similaires
entre les différentes régions.

Quelques variations sont observées, mais elles restent limitées
et sans tendance marquée.


D’un point de vue métier, la région ne semble pas constituer
un facteur de segmentation fort des charges d’assurance
dans ce dataset.


Ces résultats orienteront les choix de modélisation,
en accordant une attention particulière à la variable `smoker`
et à ses interactions potentielles avec d’autres facteurs de risque.


## 3.2 Variables numériques

### Age vs charges

Objectif :
Analyser la relation entre l’âge des assurés et les charges d’assurance
afin d’évaluer l’évolution des coûts médicaux au cours de la vie.


In [ ]:
df[["age", "charges"]].corr()


In [ ]:
plt.figure(figsize=(6,4))
sns.scatterplot(x="age", y="charges", data=df, alpha=0.6)
plt.title("Relation entre l'âge et les charges")
plt.xlabel("Âge")
plt.ylabel("Charges")
plt.show()


On observe une relation globalement croissante entre l’âge et les charges :
les coûts médicaux tendent à augmenter avec l’âge.

La corrélation est positive mais modérée, indiquant que l’âge
explique une partie de la variabilité des charges,
sans être le seul facteur déterminant.


D’un point de vue métier, cette relation est cohérente avec
l’augmentation des besoins médicaux au fil du vieillissement.

L’âge constitue donc un facteur de risque naturel
dans la tarification des contrats d’assurance.


La relation relativement linéaire entre l’âge et les charges
rend cette variable compatible avec un modèle de régression linéaire.


### Bmi vs Charges

Objectif :
Analyser l’impact de l’indice de masse corporelle (BMI)
sur les charges d’assurance afin d’évaluer le lien
entre état de santé et coûts médicaux.


In [ ]:
df[["bmi", "charges"]].corr()


In [ ]:
plt.figure(figsize=(6,4))
sns.scatterplot(x="bmi", y="charges", data=df, alpha=0.6)
plt.title("Relation entre le BMI et les charges")
plt.xlabel("BMI")
plt.ylabel("Charges")
plt.show()


La relation entre le BMI et les charges est positive mais dispersée.
Les charges tendent à augmenter pour des valeurs élevées de BMI,
mais avec une forte variabilité.

Cette dispersion suggère que le BMI seul ne suffit pas
à expliquer les charges, et que son effet dépend probablement
d’autres facteurs.


D’un point de vue métier, un BMI élevé est associé
à un risque accru de pathologies chroniques,
mais son impact financier dépend également
du profil global de l’assuré.


La variable `bmi` pourra être intégrée au modèle,
et des interactions avec le tabagisme ou l’âge
pourront être explorées afin de mieux capter son effet.


### Children vs charges

Objectif :
Analyser la relation entre le nombre d’enfants à charge
et les charges d’assurance afin d’évaluer son influence
sur les coûts médicaux.


In [ ]:
df.groupby("children")["charges"].mean()


In [ ]:
df.groupby("children")["charges"].describe()

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x="children", y="charges", data=df)
plt.title("Charges selon le nombre d'enfants")
plt.xlabel("Nombre d'enfants")
plt.ylabel("Charges")
plt.show()


Les charges moyennes ne varient pas de manière monotone
avec le nombre d’enfants.

Aucune relation claire ou linéaire ne se dégage,
suggérant un impact limité de cette variable
sur les charges individuelles.


D’un point de vue métier, le nombre d’enfants à charge
n’est pas directement lié aux coûts médicaux individuels,
mais peut refléter des caractéristiques socio-démographiques.


La variable `children` pourra être intégrée au modèle,
mais son pouvoir explicatif est attendu comme faible.


# 4 Analyse des corrélations globales

Objectif :
Analyser les relations linéaires entre les variables numériques
afin d’identifier les facteurs les plus corrélés aux charges
et d’anticiper les choix de modélisation.


In [ ]:
# Heatmap : Corrélation entre variables continues
# Visualise les corrélations entre âge, BMI, nombre d'enfants et charges.(ici nous prenons que les valeurs numériques)

df_num = df.select_dtypes(include="number")
df_num.columns

corr_matrix = df_num.corr()
corr_matrix

plt.figure(figsize=(8,6))
sns.heatmap(
    df[["age", "bmi", "children", "smoker", "charges"]].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)
plt.title("Matrice de corrélation des variables numériques")
plt.show()


En intégrant la variable `smoker`, encodée sous forme binaire (0 = non-fumeur, 1 = fumeur),
la matrice de corrélation met en évidence que le tabagisme est la variable
la plus fortement corrélée aux charges d’assurance.

La corrélation positive élevée entre `smoker` et `charges` confirme
les observations issues de l’analyse bivariée :
les assurés fumeurs génèrent en moyenne des coûts nettement plus importants.

Les variables `age` et `bmi` présentent également une corrélation positive
avec les charges, bien que d’intensité plus modérée.
Ces résultats suggèrent que l’augmentation du risque médical
est progressive avec l’âge et le niveau de BMI.

À l’inverse, la variable `children` montre une corrélation faible avec les charges,
indiquant un impact limité dans un cadre de relation linéaire.

Il est important de souligner que la corrélation calculée pour la variable `smoker`
reflète une relation moyenne entre deux groupes distincts
(fumeurs vs non-fumeurs) et ne capture pas la complexité
des interactions possibles avec d’autres variables comme l’âge ou le BMI.

Enfin, comme toute mesure de corrélation, cette analyse se limite
aux relations linéaires et ne permet pas d’inférer une relation causale.


## Analyse multivariée exploratoire (bonus)

### Interaction smoker × age

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(
    x="age",
    y="charges",
    hue="smoker_label",
    data=df,
    alpha=0.6
)
plt.title("Charges selon l'âge et le statut fumeur")
plt.show()


L’impact du tabagisme sur les charges est visible
à tous les âges, mais tend à s’amplifier
chez les assurés plus âgés.

Cela suggère une possible interaction
entre l’âge et le tabagisme.


### Interaction smoker × bmi

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(
    x="bmi",
    y="charges",
    hue="smoker_label",
    data=df,
    alpha=0.6
)
plt.title("Charges selon le BMI et le statut fumeur")
plt.show()


Chez les fumeurs, les charges augmentent fortement
avec le BMI, tandis que cette relation est
moins marquée chez les non-fumeurs.

Cette observation suggère un effet combiné
du tabagisme et du surpoids.


### a voir si on ajoute des graphe deja fais au nootebook

In [ ]:


region_counts = (
    df.groupby("region", observed=True)
      .size()
      .reset_index(name="count")
)
print(region_counts)

region_charge_sum = (
    df.groupby("region", observed=True)["charges"]
        .sum()
        .reset_index(name="total_charges")
)
print(region_charge_sum)

region_charge_mean = (
    df.groupby("region", observed=True)["charges"]
    .mean()
    .reset_index(name="mean_charges")
)
print(region_charge_mean)

region_age_mean_charges = (
    df.groupby(["region", "age"], observed=True)["charges"]
    .mean()
    .reset_index(name="mean_charges")
)
print(region_age_mean_charges)

px.bar(
    region_counts,
    x="region",
    y="count",
    title="Number of insured persons per region "
).show()

px.bar(
    region_charge_sum,
    x="region",
    y="total_charges",
    title="total charges per region "
).show()

px.bar(
    region_charge_mean,
    x="region",
    y="mean_charges",
    title="mean of charges per region "
).show()



px.line(
  region_age_mean_charges,
  x="age",
  y="mean_charges",
  color="region",
  title="Charge moyenne par âge et par region"
  ).show()

print(df["charges"])


## 3. Analyse Bivariée : Focus sur le Sexe

Vérification de l'impact de la variable `sex` sur les charges d'assurance.

In [ ]:
# Impact du sexe sur les charges
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='sex', y='charges', palette='Set2')
plt.title('Charges selon le Sexe')
plt.show()

print("Charges moyennes par sexe :")
print(df.groupby('sex', observed=True)['charges'].mean())

Analyse per sex

In [ ]:
pd.options.display.float_format = '{:,.2f}'.format

sex_counts = (
    df.groupby("sex", observed=True)
      .size()
      .reset_index(name="count")
)
print(sex_counts)

sex_charge_sum = (
    df.groupby("sex", observed=True)["charges"]
        .sum()
        .reset_index(name="total_charges")
)
print(sex_charge_sum)

sex_charge_mean = (
    df.groupby("sex", observed=True)["charges"]
    .mean()
    .reset_index(name="mean_charges")
)
print(sex_charge_mean)

sex_age_mean_charges = (
    df.groupby(["sex", "age"], observed=True)["charges"]
    .mean()
    .reset_index(name="mean_charges")
)
print(sex_age_mean_charges)

px.bar(
    sex_counts,
    x="sex",
    y="count",
    title="Number of insured persons per sex "
).show()

px.bar(
    sex_charge_sum,
    x="sex",
    y="total_charges",
    title="total charges per sex "
).show()

px.bar(
    sex_charge_mean,
    x="sex",
    y="mean_charges",
    title="mean of charges per sex "
).show()

px.line(
  sex_age_mean_charges,
  x="age",
  y="mean_charges",
  color="sex",
  title="Charge moyenne par âge et par sex"
  ).show()

px.box(
    df,
    x="sex",
    y="charges",
    title="Distribution des charges par tranche d'âge"
).show()



In [ ]:
age_counts = (
    df.groupby("age", observed=True)
      .size()
      .reset_index(name="count")
)
print(age_counts)

age_charge_sum = (
    df.groupby("age", observed=True)["charges"]
    .sum()
    .reset_index(name="total_charge")
)
print(age_charge_sum)

age_charge_mean = (
    df.groupby("age", observed=True)["charges"]
      .mean()
      .reset_index(name="mean_charge")
)
print(age_charge_mean)

age_charge_stats = df.groupby("age", observed=True)["charges"].agg(
    mean_charge="mean",
    std_charge="std",
    total_charge="sum",
    count="count"
).reset_index()
print(age_charge_stats)

bins = [18, 25, 35, 45, 55, 65]
labels = ["18-24", "25-34", "35-44", "45-54", "55-64"]
df["age_group"] = pd.cut(df["age"], bins=bins, labels=labels, right=False)





px.bar(
  age_counts,
  x="age",
  y="count", 
  title="Nombre d'assurés par âge"
  ).show()

px.bar(
  age_charge_sum,
  x="age", 
  y="total_charge",
  title="Charges totales par âge"
  ).show()

px.line(
  age_charge_mean,
  x="age",
  y="mean_charge", 
  title="Charge moyenne par âge"
  ).show()

px.box(
    df,
    x="age_group",
    y="charges",
    title="Distribution des charges par tranche d'âge"
).show()


BMI Analyse

In [ ]:
df["bmi"] = df["bmi"].astype("float64")

df["bmi_category"] = pd.cut(
    df["bmi"],
    bins=[0, 18.5, 25, 30, 100],
    labels=["Underweight", "Normal", "Overweight", "Obese"]
)

bmi_counts = (
    df.groupby("bmi_category", observed=True)
      .size()
      .reset_index(name="count")
)
print(bmi_counts)

bmi_charge_sum = (
    df.groupby("bmi_category", observed=True)["charges"]
      .sum()
      .reset_index(name="total_charges")
)
print(bmi_charge_sum)

bmi_charge_mean = (
    df.groupby("bmi_category", observed=True)["charges"]
      .mean()
      .reset_index(name="mean_charges")
)
print(bmi_charge_mean)

bmi_charge_mean_age = (
    df.groupby(["bmi_category", "age"], observed=True)["charges"]
      .mean()
      .reset_index(name="mean_charges")
)
print(bmi_charge_mean)


px.bar(
    bmi_counts,
    x="bmi_category",
    y="count",
    title="Number of insured persons per BMI category"
).show()

px.bar(
    bmi_charge_sum,
    x="bmi_category",
    y="total_charges",
    title="Total charges per BMI category"
).show()

px.bar(
    bmi_charge_mean,
    x="bmi_category",
    y="mean_charges",
    title="Mean charges per BMI category"
).show()

px.line(
  bmi_charge_mean_age,
  x="age",
  y="mean_charges",
  color="bmi_category",
  title="Charge moyenne par âge"
  ).show()

In [ ]:
bmi_mean = df["bmi"].mean()
bmi_median = df["bmi"].median()
bmi_std = df["bmi"].std()

print("Moyenne BMI :", bmi_mean)
print("Médiane BMI :", bmi_median)
print("Écart-type BMI :", bmi_std)

In [ ]:
# BMI distribution en valeur absolue

sns.histplot(df["bmi"], kde=True)
plt.title("Distribution du BMI")
plt.xlabel("BMI")
plt.ylabel("Fréquence")
plt.show()

In [ ]:
# distribution des BMI en %

sns.histplot(df["bmi"], stat="percent", kde=True)
plt.title("Distribution du BMI")
plt.xlabel("BMI")
plt.ylabel("Pourcentage (%)")
plt.show()

In [ ]:
sns.boxplot(x=df["bmi"])
plt.title("Boxplot du BMI")
plt.show()


Smokers Analyse

In [ ]:
df["smoker_label"] = df["smoker"].map({True: "Smoker", False: "No-smoker"})

smoker_counts = (
    df.groupby("smoker_label", observed=True)
      .size()
      .reset_index(name="count")
)
print(smoker_counts)

smoker_charge_sum = (
    df.groupby("smoker_label", observed=True)["charges"]
      .sum()
      .reset_index(name="total_charges")
)
print(smoker_charge_sum)

smoker_charge_mean = (
    df.groupby("smoker_label", observed=True)["charges"]
      .mean()
      .reset_index(name="mean_charges")
)
print(smoker_charge_mean)

smokers_age_mean_charges = (
    df.groupby(["smoker_label", "age"], observed=True)["charges"]
    .mean()
    .reset_index(name="mean_charges")
)
print(smokers_age_mean_charges)

df['log_charges'] = np.log(df['charges'])

sns.boxplot(x='smoker', y='log_charges', data=df)
plt.show()


px.bar(
    smoker_counts,
    x="smoker_label",
    y="count",
    title="Number of insured persons by smoking status"
).show()

px.bar(
    smoker_charge_sum,
    x="smoker_label",
    y="total_charges",
    title="Total charges by smoking status"
).show()

px.bar(
    smoker_charge_mean,
    x="smoker_label",
    y="mean_charges",
    title="Mean charges by smoking status"
).show()

px.line(
  smokers_age_mean_charges,
  x="age",
  y="mean_charges",
  color="smoker_label",
  title="Charge moyenne par âge"
  ).show()

In [ ]:
charges_smokers = df[df['smoker'] == 1]['charges']
charges_nonsmokers = df[df['smoker'] == 0]['charges']

stats_smokers = charges_smokers.describe()
stats_nonsmokers = charges_nonsmokers.describe()

print("Stats - Fumeurs")
print(stats_smokers)

print("\nStats - Non-fumeurs")
print(stats_nonsmokers)

t_stat, p_val = stats.ttest_ind(
    charges_smokers,
    charges_nonsmokers,
    equal_var=False
)

print("t-statistic:", t_stat)
print("p-value:", p_val)


mean_diff = charges_smokers.mean() - charges_nonsmokers.mean()
pooled_std = np.sqrt(
    (charges_smokers.var() + charges_nonsmokers.var()) / 2
)

cohens_d = mean_diff / pooled_std
print("Cohen's d:", cohens_d)

Interprétation :

0.2 => faible

0.5 => moyen

0.8 => fort

Ici, d > 1 = effet très fort

ici la valeur Cohen's d: 2.56807. Relation très forte entre le fait de fumer et le montant des charges.



In [ ]:
child_counts = (
    df.groupby("children")
      .size()
      .reset_index(name="count")
)
print(child_counts)

child_charge_sum = (
    df.groupby("children")["charges"]
      .sum()
      .reset_index(name="total_charges")
)
print(child_charge_sum)

child_charge_mean = (
    df.groupby("children")["charges"]
      .mean()
      .reset_index(name="mean_charges")
)
print(child_charge_mean)

child_age_mean_charges = (
    df.groupby(["children", "age"])["charges"]
    .mean()
    .reset_index(name="mean_charges")
)
print(child_age_mean_charges)

px.bar(
    child_counts,
    x="children",
    y="count",
    title="Number of insured persons with children"
).show()

px.bar(
    child_charge_sum,
    x="children",
    y="total_charges",
    title="Total charges by insured with children"
).show()


px.bar(
    child_charge_mean,
    x="children",
    y="mean_charges",
    title="Mean charges by insured with children"
).show()

child_charge_stats = df.groupby("children")["charges"].agg(
    mean_charge="mean",
    std_charge="std",
    total_charge="sum",
    count="count"
).reset_index()
print(child_charge_stats)

px.box(
    df,
    x="children",
    y="charges",
    title="distribution of burdens by age groups"
).show()

px.line(
  child_age_mean_charges,
  x="age",
  y="mean_charges",
  color="children",
  title="Charge moyenne par âge"
  ).show()

In [ ]:
# en %
df["smoker"].value_counts(normalize=True) * 100

In [ ]:
# Mediane et mean des charges pr fumeurs ou non

df.groupby("smoker")["charges"].agg(["count", "mean", "median"])

In [ ]:
sns.boxplot(data=df, x="smoker", y="charges")
plt.title("Charges selon le statut fumeur")
plt.xlabel("Fumeur")
plt.ylabel("Charges")
plt.show()

Fumeurs et Bmi

In [ ]:
sns.scatterplot(data=df, x="bmi", y="charges", hue="smoker")
plt.title("Relation BMI vs Charges (par statut fumeur)")
plt.xlabel("BMI")
plt.ylabel("Charges")
plt.show()

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
sns.histplot(df['charges'], bins=50, kde=True)
plt.title("Distribution des charges")

plt.subplot(1,2,2)
sns.boxplot(x=df['charges'])
plt.title("Boxplot des charges")

plt.show()


In [ ]:
quantiles = df['charges'].quantile([0.90, 0.95, 0.99])
print(quantiles)

total_charges = df['charges'].sum()

for q in [0.95, 0.99]:
    threshold = df['charges'].quantile(q)
    part = df[df['charges'] >= threshold]['charges'].sum() / total_charges
    print(f"Part des charges au-dessus du quantile {q*100:.0f}% : {part:.2%}")


5% des assurer 17.58% des charges totales

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Optionnel : style seaborn
sns.set(style="whitegrid")


In [ ]:
# 1️ Histogramme des charges
# Montre la distribution des charges d’assurance.

plt.figure(figsize=(8,4))
sns.histplot(df['charges'], bins=50, kde=True)
plt.title("Distribution des charges")
plt.show()









## 1️Histogramme des charges


**Observation**  
- La majorité des charges se situe autour de **10 000 $**.  
- Quelques **outliers très élevés** dépassant **50 000 $**.  
- Distribution **asymétrique à droite**.

**Conclusion**  
Certaines valeurs extrêmes peuvent **influencer un modèle**.  
Une **transformation logarithmique** est à envisager pour **réduire l’impact des outliers**.


In [ ]:

# 2️ Histogramme du BMI
# Montre la distribution de l’indice de masse corporelle (BMI).

sns.histplot(df['bmi'], bins=30, kde=True)
plt.title("Distribution du BMI")
plt.show()


## 2️ Histogramme du BMI

**Observation**  
- La plupart des assurés ont un **BMI entre 20 et 35**.  
- Quelques **valeurs extrêmes plausibles**.

**Conclusion**  
Les valeurs sont **cohérentes** et ne nécessitent pas de nettoyage,  
mais il est important de **noter la présence de quelques outliers**.


In [ ]:
# 3️ Histogramme de l’âge
# Montre la répartition des âges.
plt.figure(figsize=(8,4))
sns.histplot(df['age'], bins=30, kde=True)
plt.title("Distribution de l'âge")
plt.show()

**Observation**  
- Âge compris entre **18 et 64 ans**.  
- Distribution **assez normale**.

**Conclusion**  
Variable **propre et cohérente**, sans **valeurs aberrantes**.


In [ ]:
# 1️Boxplot des charges
# Visualise la médiane, les quartiles et les outliers pour les charges.


sns.boxplot(x=df['charges'])
plt.title("Boxplot des charges")
plt.show()










## 1️Boxplot des charges

**Objectif**  
Visualiser la **médiane**, les **quartiles** et les **outliers** des charges.

**Observation**  
- La majorité des charges est concentrée autour de **10 000 $**.  
- Présence de **plusieurs outliers très élevés**, dépassant **50 000 $**.

**Conclusion**  
Ces valeurs extrêmes peuvent **fortement influencer un modèle linéaire**.  
Une **transformation logarithmique** des charges peut être envisagée afin de **réduire l’impact des outliers** et améliorer la stabilité du modèle.


In [ ]:
#2️ Boxplot du BMI
# Montre la distribution du BMI avec médiane, quartiles et outliers.

sns.boxplot(x=df['bmi'])
plt.title("Boxplot du BMI")
plt.show()

## 2️ Boxplot du BMI

**Objectif**  
Montrer la distribution du **BMI** avec médiane, quartiles et outliers.

**Observation**  
- La plupart des individus ont un **BMI entre 20 et 35**.  
- Quelques **valeurs extrêmes au-delà de 50**.

**Conclusion**  
Les valeurs sont **plausibles**, mais les **outliers doivent être pris en compte** pour l’analyse et éventuellement pour le **scaling**.


In [ ]:
# 3️ Boxplot de l'âge
# Visualise la répartition de l'âge.


sns.boxplot(x=df['age'])
plt.title("Boxplot de l'âge")
plt.show()


## 3️ Boxplot de l’âge

**Objectif**  
Visualiser la répartition de l’âge.

**Observation**  
- Âge compris entre **18 et 64 ans**.  
- Distribution **plutôt normale**, avec **peu d’outliers**.

**Conclusion**  
Variable **propre et cohérente**, ne nécessitant **pas de nettoyage particulier**.


In [ ]:
# 4️ Boxplot Charges vs Tabagisme
# Compare les charges entre fumeurs et non-fumeurs.

sns.boxplot(x='smoker', y='charges', data=df)
plt.title("Charges vs Tabagisme")
plt.show()


## 4️Boxplot Charges vs Tabagisme

**Objectif**  
Comparer les charges entre **fumeurs** et **non-fumeurs**.

**Observation**  
- Les **fumeurs** ont des charges **beaucoup plus élevées** que les non-fumeurs.  
- Présence d’**outliers encore plus extrêmes** chez les fumeurs.

**Conclusion**  
Le **tabagisme** est un **facteur clé** pour expliquer les charges.  
Cette variable doit **absolument être incluse dans le modèle**.

```python
sns.boxplot(x='smoker', y='charges', data=df)
plt.title("Charges vs Tabagisme")
plt.show()


In [ ]:
# 1️Scatterplot Charges vs Age
# Visualise la relation entre l'âge et les charges, en différenciant fumeurs et non-fumeurs.


sns.scatterplot(x='age', y='charges', hue='smoker', data=df)
plt.title("Charges vs Age selon tabagisme")
plt.show()





## 1️ Scatterplot Charges vs Âge

**Objectif**  
Visualiser la relation entre l’âge et les charges, en distinguant les **fumeurs** et les **non-fumeurs**.

**Observation**  
- Pour les **non-fumeurs**, les charges restent relativement **faibles** et varient peu avec l’âge.  
- Pour les **fumeurs**, les charges **augmentent avec l’âge** et présentent des **outliers très élevés**.

**Conclusion**  
L’effet de l’âge sur les charges dépend fortement du **tabagisme**.  
L’interaction **âge × smoker** est donc **essentielle à intégrer dans le modèle** afin d’améliorer la qualité des prédictions.


In [ ]:
#2️ Scatterplot Charges vs BMI
# Visualise la relation entre le BMI et les charges, en différenciant fumeurs et non-fumeurs.


sns.scatterplot(x='bmi', y='charges', hue='smoker', data=df)
plt.title("Charges vs BMI selon tabagisme")
plt.show()


In [ ]:
sns.scatterplot(x='age', y='charges', hue='smoker', data=df)
plt.title("Charges vs Age selon tabagisme")
plt.show()

## 2️ Scatterplot Charges vs BMI

**Objectif**  
Visualiser la relation entre le **BMI** et les charges, en distinguant les **fumeurs** et les **non-fumeurs**.

**Observation**  
- Pour les **non-fumeurs**, les charges sont **modérées** avec une **légère augmentation** lorsque le BMI augmente.  
- Pour les **fumeurs**, les charges **augmentent beaucoup plus fortement** avec le BMI, avec la présence de **plusieurs outliers extrêmes**.

**Conclusion**  
Le **BMI** a un impact principalement chez les **fumeurs**.  
L’interaction **BMI × smoker** constitue donc un **facteur clé** pour la prédiction des charges.


In [ ]:
# 1️ Répartition fumeurs / non-fumeurs
# Visualise le nombre d'individus fumeurs et non-fumeurs.

sns.countplot(x='smoker', data=df)
plt.title("Répartition fumeurs / non-fumeurs")
plt.show()





## 1️ Répartition fumeurs / non-fumeurs

**Objectif**  
Visualiser la répartition des individus fumeurs et non-fumeurs dans le jeu de données.

**Observation**  
- Environ **80 %** des assurés sont **non-fumeurs**.  
- Environ **20 %** des assurés sont **fumeurs**.

**Conclusion**  
La variable **`smoker`** est **déséquilibrée**, mais elle est **très déterminante** pour expliquer le montant des charges.  
Elle devra impérativement être **incluse dans le modèle**, car elle est susceptible d’avoir une **influence forte sur la prédiction**.


In [ ]:
# 2️ Répartition des assurés par région
# Visualise le nombre d'individus dans chaque région.


sns.countplot(x='region', data=df)
plt.title("Répartition des assurés par région")
plt.show()


## 2️ Répartition des assurés par région

**Objectif**  
Visualiser le nombre d’individus dans chaque région.

**Observation**  
- Les quatre régions (**northeast**, **northwest**, **southeast**, **southwest**) sont **relativement équilibrées**.  
- Chaque région compte environ **320 à 360 assurés**.

**Conclusion**  
Il existe **peu de biais lié à la région**.  
Cette variable peut être utilisée pour des **analyses groupées** ou intégrée comme **variable catégorielle** dans le modèle.


### 4. Matrix de Corrélation

**Note Technique :** Seuses les variables numériques (`age`, `bmi`, `children`, `charges`) sont incluses dans cette heatmap. 

Les variables catégorielles (`sex`, `smoker`, `region`) sont exclues à ce stade car elles nécessitent un encodage (ex: One-Hot Encoding ou Label Encoding) pour être traitées mathématiquement par la corrélation de Pearson. Cette étape sera réalisée en Semaine 2 lors de la préparation des données.

### Justification de la sélection des variables

Pour cette analyse de corrélation, nous avons sélectionné uniquement les variables **numériques** :
- `age` : âge de l'assuré
- `bmi` : indice de masse corporelle
- `children` : nombre d'enfants
- `charges` : frais médicaux (variable cible)

**Pourquoi `smoker` et `region` ne sont pas incluses ?**
- Ce sont des **variables catégorielles** (qualitatives)
- La corrélation de Pearson mesure uniquement les relations **linéaires** entre variables **numériques**
- Pour analyser ces variables, il faudra les **encoder** (transformer en nombres) → ce sera fait en semaine 2

###  Limites de la corrélation linéaire

1. **Ne capture que les relations linéaires**
   - Si deux variables ont une relation non-linéaire (ex: quadratique), la corrélation de Pearson peut être faible
   - Exemple : `bmi` pourrait avoir un effet non-linéaire sur `charges`

2. **Corrélation ≠ Causalité**
   - Une forte corrélation ne signifie pas qu'une variable **cause** l'autre
   - Il peut y avoir des variables cachées (confondantes)

3. **Sensible aux valeurs extrêmes**
   - Les outliers peuvent fausser les coefficients de corrélation

4. **Variables catégorielles exclues**
   - `smoker` et `region` ne sont pas analysées ici
   - Pourtant, `smoker` pourrait être **très important** pour prédire `charges`

In [ ]:
# Heatmap : Corrélation entre variables continues
# Visualise les corrélations entre âge, BMI, nombre d'enfants et charges.
plt.figure(figsize=(6,4))
sns.heatmap(df[['age','bmi','children','charges']].corr(), annot=True, cmap='coolwarm')
plt.title("Corrélation entre variables continues")
plt.show()



In [ ]:

print("🔎 Aperçu global du dataset")
print(f"Nombre d'observations : {df.shape[0]}")
print(f"Nombre de variables : {df.shape[1]}")

print("\n📈 Statistiques clés sur les charges (charges)")
charges_mean = df["charges"].mean()
charges_median = df["charges"].median()
charges_std = df["charges"].std()

print(f"Charge moyenne   : {charges_mean:.2f}")
print(f"Charge médiane   : {charges_median:.2f}")
print(f"Écart-type       : {charges_std:.2f}")

print("\n👥 Analyse par catégorie")

# Charges moyennes par statut fumeur
print("\nCharges moyennes par statut fumeur :")
print(df.groupby("smoker")["charges"].mean().round(2))

print("\n📊 Corrélations numériques")
numeric_cols = df.select_dtypes(include=np.number)
print(numeric_cols.corr()["charges"].sort_values(ascending=False).round(2))

print("\n🧠 Key Insights")
print("""
1. Le statut de fumeur est le facteur le plus discriminant sur le montant des charges.
2. L'âge est positivement corrélé avec les charges.
3. Le BMI a un impact modéré mais non négligeable.
4. Le nombre d'enfants a un impact limité.
5. Les différences régionales existent mais restent secondaires.
""")

###  Heatmap – Corrélation entre variables continues

Cette visualisation met en évidence les corrélations entre les variables numériques :
**âge**, **BMI**, **nombre d’enfants** et **charges**.

#### Observations
- Les **charges** corrèlent positivement avec :
  - l’**âge** (corrélation ≈ 0.3),
  - le **BMI** (corrélation ≈ 0.2–0.3).
- La corrélation entre les **charges** et le **nombre d’enfants** est très faible
  (corrélation ≈ 0.1), indiquant un impact limité.
- Les corrélations entre les autres variables continues restent faibles à modérées.

#### Conclusion
- Les variables continues les plus importantes pour expliquer les **charges**
  sont l’**âge** et le **BMI**.
- La variable **children** a un effet marginal et pourrait être moins influente
  dans le modèle.
- Cette heatmap permet d’identifier les variables prioritaires pour la phase de
  modélisation et d’orienter la sélection de features.


##  Synthèse finale – Analyse exploratoire (Semaine 1)

### Qualité et structure des données
- Le jeu de données est globalement propre.
- Aucune valeur manquante critique n’a été détectée.
- Un nombre très limité de doublons a été identifié et traité.
- Les types de variables sont cohérents :
  - **Numériques** : `age`, `bmi`, `children`, `charges`
  - **Catégorielles** : `sex`, `smoker`, `region`

---

### Analyse des distributions

**Charges**
- Distribution fortement asymétrique à droite avec plusieurs valeurs extrêmes (> 50 000$).
- Ces outliers peuvent fortement influencer un modèle linéaire.
- Une transformation logarithmique des charges est pertinente pour la modélisation.

**BMI**
- La majorité des valeurs est comprise entre 20 et 35.
- Quelques valeurs extrêmes existent mais restent plausibles.
- Variable exploitable telle quelle, en conservant les outliers à l’esprit.

**Âge**
- Distribution cohérente entre 18 et 64 ans.
- Pas de valeurs aberrantes significatives.
- Variable stable et bien adaptée à la modélisation.

---

### Relations entre variables
- Le **tabagisme** est le facteur le plus discriminant :
  - Les fumeurs ont des charges nettement plus élevées que les non-fumeurs.
- L’**âge** et le **BMI** ont un impact positif sur les charges,
  - impact beaucoup plus marqué chez les fumeurs.
- Le **nombre d’enfants** présente une corrélation faible avec les charges.
- Les **régions** sont relativement équilibrées et expliquent peu la variance des charges.

---

### Enseignements pour la suite du projet
- Variables clés pour la modélisation :
  - `smoker`, `age`, `bmi`
- Présence d’outliers et forte asymétrie des charges :
  - nécessité d’un **scaling**
  - justification d’une **transformation logarithmique**
- Ces observations justifient la mise en place d’un **pipeline de préparation des données**
  avant la phase de modélisation (Semaine 2 : baseline et évaluation).


In [ ]:
from ydata_profiling import ProfileReport

# Création du rapport
profile = ProfileReport(
    df,
    title="Insurance Dataset – EDA Report",
    explorative=True
)

# Affichage du rapport
profile



In [ ]:
from ydata_profiling import ProfileReport

# Génération du rapport automatique (Bonus)
profile = ProfileReport(df, title="Rapport EDA Assurance", explorative=True)
profile.to_file("insurance_eda_report.html")
print("Rapport 'insurance_eda_report.html' généré avec succès.")